# Day D7 — 패턴 분석과 ROI: decoding, 교차검증 누수, circularity, % 신호 변화

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/D-fmri/day-D7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/D-fmri/day-D7.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q nilearn

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 신호 없는 자료로 98%가 나오는 누수, 그리고 올바른 파이프라인

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score, permutation_test_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n_runs, per_run, n_vox = 6, 10, 3000
y = np.tile([0, 1], n_runs * per_run // 2)              # 두 조건이 번갈아 나오는 시행 60개
runs = np.repeat(np.arange(n_runs), per_run)             # 시행이 어느 run에서 나왔는지
X_null = rng.normal(size=(y.size, n_vox))                # 신호가 '전혀 없는' 시행별 voxel 패턴

clf = LogisticRegression(C=1.0, max_iter=2000)
cv = GroupKFold(n_splits=n_runs)                         # run 하나씩 통째로 빼서 시험

# 잘못된 방법: 전체 자료로 '좋은 voxel' 50개를 먼저 고른 뒤 교차검증
top = SelectKBest(f_classif, k=50).fit(X_null, y).get_support()
leaky = cross_val_score(make_pipeline(StandardScaler(), clf), X_null[:, top], y, cv=cv, groups=runs)

# 올바른 방법: voxel 고르기를 pipeline 안에 넣어 학습 fold에서만 하게 한다
pipe = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=50), clf)
honest = cross_val_score(pipe, X_null, y, cv=cv, groups=runs)
print("신호 없음 | 고르고 나서 CV: %.2f | CV 안에서 고르기: %.2f" % (leaky.mean(), honest.mean()))

# 진짜 신호가 약하게 있는 경우 + 정답 섞기(permutation)로 우연 분포 만들기
X_sig = X_null.copy()
X_sig[:, :30] += 0.8 * (y[:, None] - 0.5)                 # 30개 voxel에만 작은 차이
score, perm, p = permutation_test_score(pipe, X_sig, y, cv=cv, groups=runs,
                                        n_permutations=200, random_state=0)
print("약한 신호 | 정확도 %.2f, 섞은 정답 정확도 평균 %.2f (95%%: %.2f), p = %.3f"
      % (score, perm.mean(), np.quantile(perm, 0.95), p))

### 예시 2 — 같은 자료로 고르고 재면 효과가 부풀려진다

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(12)
n_sub, n_vox = 20, 5000
true_effect = np.zeros(n_vox); true_effect[:100] = 0.3     # 100개 voxel에만 작은 진짜 효과(% 신호 변화)
behavior = rng.normal(size=n_sub)                            # 가상 행동 점수 (뇌와 무관)

def run_once():
    # 두 번의 독립 측정(예: 홀수 run / 짝수 run)
    half1 = true_effect + rng.normal(0, 1, (n_sub, n_vox))
    half2 = true_effect + rng.normal(0, 1, (n_sub, n_vox))
    t1 = stats.ttest_1samp(half1, 0).statistic
    roi = t1 > stats.t.isf(0.001, n_sub - 1)                # half1에서 '유의한' voxel을 고른다
    if roi.sum() == 0:
        return None
    circular = half1[:, roi].mean()                          # 고른 그 자료로 효과 크기 재기
    independent = half2[:, roi].mean()                       # 다른 자료로 재기
    # 행동 상관도 같은 함정: 고를 때 상관이 높은 voxel을 고르면 상관이 부풀려진다
    r_all = np.array([np.corrcoef(behavior, half1[:, v])[0, 1] for v in range(n_vox)])
    best = np.argsort(r_all)[-10:]                           # 상관이 가장 높은 voxel 10개
    r_circ = np.corrcoef(behavior, half1[:, best].mean(1))[0, 1]
    r_ind = np.corrcoef(behavior, half2[:, best].mean(1))[0, 1]
    return roi.sum(), circular, independent, r_circ, r_ind, true_effect[roi].mean()

out = np.array([o for o in (run_once() for _ in range(50)) if o is not None])
print("고른 voxel 수(평균): %.0f" % out[:, 0].mean())
print("고른 voxel들의 진짜 효과 평균: %.2f (가짜로 뽑힌 voxel이 섞여 0.30보다 작다)" % out[:, 5].mean())
print("효과 크기 추정 — 같은 자료: %.2f | 독립 자료: %.2f" % (out[:, 1].mean(), out[:, 2].mean()))
print("뇌–행동 상관 — 진짜 0 | 같은 자료: %.2f | 독립 자료: %.2f" % (out[:, 3].mean(), out[:, 4].mean()))

### 예시 3 — % 신호 변화: 고정된 "자"와 contrast 배율

In [ ]:
import numpy as np
from scipy.stats import gamma

dt = 0.1
t = np.arange(0, 32, dt)
hrf = gamma.pdf(t, 6) - gamma.pdf(t, 16) / 6

def unit_regressor(dur, total=60):
    stim = np.zeros(int(total / dt)); stim[: int(dur / dt)] = 1
    return np.convolve(stim, hrf)[: stim.size] * dt     # 정규화하지 않은 '그대로의' 높이

ruler = unit_regressor(1.0).max()       # 기준 사건(1초 자극 하나)의 봉우리 높이 = '자'
print("1초 자극 회귀변수의 높이(자): %.4f" % ruler)

def percent_signal_change(beta, baseline_mean, scale=1.0):
    return 100 * scale * beta * ruler / baseline_mean

beta_face, beta_house, mean_bold = 9.5, 6.0, 820.0     # 가상 GLM 결과와 시계열 평균
print("얼굴 PSC: %.3f%%" % percent_signal_change(beta_face, mean_bold))
# contrast [1, -1]과 [2, -2]는 같은 가설이지만 값의 크기가 두 배 다르다 → 배율(scale)로 맞춘다
for c in ([1, -1], [2, -2]):
    con = c[0] * beta_face + c[1] * beta_house
    sf = 1 / sum(v for v in c if v > 0)                 # 양수 가중치 합이 1이 되도록
    print(f"contrast {c}: 그대로 {percent_signal_change(con, mean_bold):.3f}%"
          f" → 배율 보정 {percent_signal_change(con, mean_bold, sf):.3f}%")

### 참고 — nilearn으로 run 단위 decoding을 할 때의 형태 (nilearn 필요)

> 이 코드는 `pip install nibabel nilearn`으로 설치한 환경(nibabel 5.5, nilearn 0.14.1)에서, 같은 이름의 **가상 파일**을 만들어 그대로 실행해 오류 없이 도는 것을 확인했습니다. 출력은 여러분의 실제 파일에 따라 달라지므로 싣지 않았습니다.

In [ ]:
# nilearn이 필요하다: pip install nilearn
from nilearn.decoding import Decoder
from sklearn.model_selection import LeaveOneGroupOut
decoder = Decoder(estimator="svc", mask=roi_mask_img, standardize="zscore_sample",
                  screening_percentile=5, cv=LeaveOneGroupOut(), scoring="balanced_accuracy")
decoder.fit(trial_beta_imgs, conditions, groups=run_labels)   # voxel 선별은 fold 안에서 일어난다
print(decoder.cv_scores_)